In [15]:
# The imports
import pandas as pd
import numpy as np

In [16]:
# Import the parsed dataframe to this notebook
data = pd.read_parquet("data/processed/01_inpatient_activity_clean.parquet")

In [17]:
# Create a "subset" dataFrame that points to the rows where at least one value is NA
na_mask = data.isna().any(axis=1)
missing_vals = data.loc[na_mask]

In [18]:
fig1_data = data.groupby(['Region', 'Month'])[['Discharges', 'Bed days']].sum()
fig1_data.reset_index(inplace=True) # To be able to group by the dates' month value in the next lines

# Create new columns that store the maximum and minimum bed days and discharge values reached in each month throughout the years
fig1_data["Discharge max monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Discharges'].transform("max")
fig1_data["Discharge min monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Discharges'].transform("min")
fig1_data["Bed days max monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Bed days'].transform("max")
fig1_data["Bed days min monthly value"] = fig1_data.groupby(['Region', fig1_data['Month'].dt.month])['Bed days'].transform("min")

In [19]:
fig2_data = data.groupby(['Region', 'Month'])[['Discharges', 'Bed days']].sum()
fig2_data.reset_index(inplace=True) # To be able to group by the dates' month value in the next lines

# Create a new column that stores the median bed days and discharge values reached in each month throughout the years
fig2_data["Discharge median monthly value"] = fig2_data.groupby(['Region', fig2_data['Month'].dt.month])['Discharges'].transform(np.median)
fig2_data["Bed days median monthly value"] = fig2_data.groupby(['Region', fig2_data['Month'].dt.month])['Bed days'].transform(np.median)

fig2_data = fig2_data.query('Month.dt.year == 2025')

In [20]:
fig3_data = (
    data
    .groupby(['Region', 'Specialty type'])  
    .resample('QE')                         
    [['Discharges', 'Bed days']]
    .sum()
)

fig3_data = fig3_data.swaplevel(2, 1)

In [21]:
# Intermediate tables to help fetch the required values with more readable code syntax
surg_data = data[data["Specialty type"] == "Especialidade Cirurgica"]
med_data = data[data["Specialty type"] == "Especialidade Médica"]
other_data = data[data["Specialty type"] == "Outras Camas"]

annual_data = data.groupby("Region")[['Bed days', 'Discharges']].resample("YE").sum()
annual_data.reset_index(inplace = True)
annual_data.rename(columns = {'Month': "Year"}, inplace = True)

# Build the dataframe with the required values.
final_table_data = (   # Expression on multiple lines for enhanced readability; uses the concept of named aggregation, with plain tuples. Found the feature at https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation
    data
    .groupby('Region')
    .agg(
        mean_bed_days=('Bed days', 'mean'),
        std_bed_days=('Bed days', 'std'),
        mean_discharges=('Discharges', 'mean'),
        std_discharges=('Discharges', 'std'),
    )
)

final_table_data['coef bed days'] = final_table_data['std_bed_days'] / final_table_data['mean_bed_days']
final_table_data['coef discharges'] = final_table_data['std_discharges'] / final_table_data['mean_discharges']

final_table_data['Surgical total bed days'] = surg_data.groupby("Region")["Bed days"].sum()
final_table_data['Medical total bed days'] = med_data.groupby("Region")["Bed days"].sum()
final_table_data['Other total bed days'] = other_data.groupby("Region")["Bed days"].sum()

final_table_data['Surgical total discharges'] = surg_data.groupby("Region")["Discharges"].sum()
final_table_data['Medical total discharges'] = med_data.groupby("Region")["Discharges"].sum()
final_table_data['Other total discharges'] = other_data.groupby("Region")["Discharges"].sum()

final_table_data['Surg bed days per discharge'] = final_table_data['Surgical total bed days'] / final_table_data['Surgical total discharges']
final_table_data['Med bed days per discharge'] = final_table_data['Medical total bed days'] / final_table_data['Medical total discharges']
final_table_data['Other bed days per discharge'] = final_table_data['Other total bed days'] / final_table_data['Other total discharges']

coef_var_data = annual_data[annual_data["Year"] # This was a challenging line to code. Initially, the "in" boolean operator was tested, however it would not work as intended.  The "isin" pandas method was then discoved through research.
                                # Such method was not even mentioned in the "Python for Data Analysis" book, which underlines that no amount of theory will prepare you for every scenario in programming projects.
                                # The newlines were inserted in this statement to enhance human readability.
            .isin(
                [pd.Period("2015", freq="Y"), 
                 pd.Period("2025", freq="Y")
                ]
            )
]

In [ ]:
# Save the dataFrames to a file
fig1_data.to_parquet("data/processed/02_fig1_data.parquet")
fig2_data.to_parquet("data/processed/02_fig2_data.parquet")
fig3_data.to_parquet("data/processed/02_fig3_data.parquet")
final_table_data.to_parquet("data/processed/02_final_table_data.parquet")
coef_var_data.to_parquet("data/processed/02_coef_var_data.parquet")